# Bronze 2: split the long PDFs into parts

Task `bronze_split` of the job `ingest_financebench`. Cuts every PDF in `bronze_pdfs` that has more than `part_pages` pages into **parts** of at most `part_pages` pages, and stores the bytes of each part in `bronze_pdf_parts`. A PDF with `part_pages` pages or fewer is one part. Silver reads the parts, never the whole PDFs.

```
bronze_pdfs (whole PDFs)  -->  bronze_pdf_parts (path, part_no, page_offset, pages, content)
```

**Why.** The parser throttles (`Request rate limit exceeded`) when it gets a long PDF in one request, and a failed page throws away the whole parse. In a test, a 155-page filing lost 27 pages when parsed whole, and came back complete in four parts of 50, 50, 50 and 5 pages. A part that fails is retried alone, so a retry costs at most `part_pages` pages. The parser also rejects PDFs over 500 pages: as parts they can be parsed. Cutting is not a transformation of the content, so it belongs in bronze: the parts are still raw bytes.

| Rule | Why |
| --- | --- |
| Parts are stored in a table, not as files | Nothing to track in the volume, and a part keeps its original `path` and `page_offset` |
| Only PDFs without parts are cut, one PDF at a time | Running it twice does nothing the second time. Reading all the PDFs at once through the driver failed (`UnknownException`) on the real data |
| A PDF that cannot be read here is one part with its original bytes | The parser decides, and a failure is logged like any other |

The real page number of a page is `page_offset + page_id + 1`, so the pages, the log and the documents keep the page numbers of the original PDF. The cutting logic is in [splitting.py](./splitting.py) (tested with `pytest`).

In [ ]:
%pip install pypdf -q

In [ ]:
dbutils.library.restartPython()

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "../common")
from pyspark.sql import functions as F, types as T

from names import Names
from bronze_tables import create_parts_table

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("bronze_schema", "finance_bench_bronze")
dbutils.widgets.text("part_pages", "50")  # PDFs with more pages are cut into parts of this size

n = Names(dbutils.widgets.get("catalog"), dbutils.widgets.get("bronze_schema"), "")
part_pages = int(dbutils.widgets.get("part_pages"))

create_parts_table(spark, n)

# splitting.py is next to this notebook, so the executors cannot import it: send the file to them.
spark.addArtifacts(str(Path("splitting.py").resolve()), pyfile=True)

parts_type = T.ArrayType(T.StructType([
    T.StructField("part_no", T.IntegerType()), T.StructField("page_offset", T.IntegerType()),
    T.StructField("pages", T.IntegerType()), T.StructField("content", T.BinaryType()),
]))


@F.udf(returnType=parts_type)
def cut(content, size):
    """The parts of one PDF: (part_no, page_offset, pages, bytes). Runs on an executor."""
    import splitting

    return splitting.split_pdf(bytes(content), size)

In [ ]:
# The PDFs that have no parts yet. One PDF per task: the cutting runs in parallel on the executors, and the
# bytes go from the executors straight into the table, never through the driver.
todo = spark.table(n.bronze).select("path", "content").join(spark.table(n.parts).select("path").distinct(), "path", "left_anti")
pdfs = todo.select("path").count()
print(f"{pdfs} PDFs to split.")

if pdfs:
    (
        todo.repartition(pdfs)
        .select("path", F.explode(cut("content", F.lit(part_pages))).alias("p"))
        .select("path", "p.part_no", "p.page_offset", "p.pages", "p.content", F.current_timestamp().alias("split_at"))
        .write.mode("append")
        .option("optimizeWrite", "false")  # one file per PDF (not merged)
        .saveAsTable(n.parts)
    )

In [ ]:
display(spark.sql(f"SELECT count(DISTINCT path) AS pdfs, count(*) AS parts, max(pages) AS longest_part FROM {n.parts}"))
print("files in", n.parts, ":", spark.sql(f"DESCRIBE DETAIL {n.parts}").first().numFiles)